In [1]:
from transformers import pipeline


pipeline = pipeline(
    task="fill-mask",
    model="answerdotai/ModernBERT-base",
    device=0
)
pipeline("Plants create [MASK] through a process known as photosynthesis.")

/Users/vignesh/Documents/personal/OpenJev/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 137/137 [00:00<00:00, 18295.91it/s]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'score': 0.5907037854194641,
  'token': 2739,
  'token_str': ' food',
  'sequence': 'Plants create food through a process known as photosynthesis.'},
 {'score': 0.08836915343999863,
  'token': 7768,
  'token_str': ' oxygen',
  'sequence': 'Plants create oxygen through a process known as photosynthesis.'},
 {'score': 0.06198470667004585,
  'token': 8383,
  'token_str': ' glucose',
  'sequence': 'Plants create glucose through a process known as photosynthesis.'},
 {'score': 0.05297405645251274,
  'token': 36984,
  'token_str': ' sugars',
  'sequence': 'Plants create sugars through a process known as photosynthesis.'},
 {'score': 0.04515177011489868,
  'token': 2341,
  'token_str': ' energy',
  'sequence': 'Plants create energy through a process known as photosynthesis.'}]

In [2]:
import os
import sys

# Run from the repo root so `import openjev` and checkpoints/ resolve.
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebook" else os.getcwd()
os.chdir(ROOT)
sys.path.insert(0, ROOT)

from openjev import OpenJev, OpenJevPipeline

# Uses checkpoints/openjev if you've trained one; otherwise ModernBERT-base with
# untrained heads (warns, and the probabilities are meaningless until you train).
jev = OpenJev.load("openjev")
pipe = OpenJevPipeline(model=jev.model, tokenizer=jev.tokenizer, device=jev.device)
pipe

/Users/vignesh/Documents/personal/OpenJev/openjev/infer.py:80: UserWarning: No trained OpenJev checkpoint for 'openjev'; the scoring heads are untrained and their probabilities are meaningless. Train first with scripts/train.py.
  warnings.warn(
Loading weights: 100%|██████████| 134/134 [00:00<00:00, 9539.30it/s]
[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


OpenJevPipeline(encoder='answerdotai/ModernBERT-base', device=mps)

In [3]:
# One question at a time, one of each type.
print(pipe({"type": "noul", "state": "The customer says the item arrived broken."}))
print(pipe({"type": "choice", "state": "Refund request for a damaged item",
            "options": ["approve", "deny", "escalate"]}))
print(pipe({"type": "multi", "state": "Arrived late and the box was crushed",
            "options": ["late", "damaged", "wrong item"]}, threshold=0.5))

{'type': 'noul', 'noul': 0.5082672834396362}
{'type': 'choice', 'choice': 'approve', 'probabilities': {'approve': 0.3338516056537628, 'deny': 0.3332124650478363, 'escalate': 0.33293604850769043}, 'confidence': 5.913901335308225e-07}
{'type': 'multi', 'selected': [], 'probabilities': {'late': 0.26624852418899536, 'damaged': 0.2854210138320923, 'wrong item': 0.30305638909339905}}


In [4]:
import math

# A mixed list, batched, checked against OpenJev.ask_batch on the same questions.
questions = [
    {"type": "noul", "state": "The package was delivered on time."},
    {"state": "Customer wants to cancel before shipping", "options": ["cancel", "keep", "escalate"]},
    {"type": "multi", "state": "Wrong color and missing charger", "options": ["wrong item", "missing part", "late"]},
    {"state": "Card was charged twice", "options": ["refund", "deny"]},
]

batched = pipe(questions, batch_size=3)
expected = jev.ask_batch(questions)

for got, want in zip(batched, expected):
    print(got)
    got_p = got.get("probabilities", {"noul": got.get("noul")})
    want_p = want.get("probabilities", {"noul": want.get("noul")})
    assert all(math.isclose(got_p[k], want_p[k], abs_tol=1e-4) for k in want_p), (got, want)
print("pipeline matches OpenJev.ask_batch")

{'type': 'noul', 'noul': 0.49430203437805176}
{'type': 'choice', 'choice': 'keep', 'probabilities': {'cancel': 0.3287544548511505, 'keep': 0.38607510924339294, 'escalate': 0.2851705253124237}, 'confidence': 0.0069727142015877375}
{'type': 'multi', 'selected': [], 'probabilities': {'wrong item': 0.3587959408760071, 'missing part': 0.34663957357406616, 'late': 0.2899135649204254}}
{'type': 'choice', 'choice': 'refund', 'probabilities': {'refund': 0.5096225738525391, 'deny': 0.49037742614746094}, 'confidence': 0.0002671860948108007}
pipeline matches OpenJev.ask_batch


In [ ]:
# A generator streams: results come back one by one as batches finish.
stream = (q for q in questions)
for result in pipe(stream, batch_size=2):
    print(result)